# 04 — DuckDB complet

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre ce qu'est DuckDB et quand l'utiliser;
- exécuter du SQL directement sur des fichiers CSV/Parquet/JSON;
- intégrer DuckDB avec Pandas et Polars (interop zero-copy);
- écrire des jointures, agrégations, window functions;
- utiliser les **extensions** DuckDB (httpfs, spatial, icu);
- choisir entre une base persistante et une base en mémoire;


## Prérequis — ce que vous devez maîtriser

Ce notebook fait partie des **Side tracks**. Il est conçu pour être autonome, à condition d'avoir suivi le niveau **Intermédiaire** de la formation principale. Vous devez donc être à l'aise avec :

- SQL de base (`SELECT`, `WHERE`, `GROUP BY`, `JOIN`);
- Pandas ou Polars, pour l'interop;
- notions de Parquet (colonnaire);

Ce que ce notebook **ne couvrira pas** (ou seulement en mention) :

- l'administration d'un serveur de base classique (DuckDB est embarqué);
- la modélisation dimensionnelle poussée;


> **Note.** DuckDB est une base de données analytique **embarquée** (comme SQLite) mais conçue pour les requêtes OLAP. Elle lit Parquet/CSV en zero-copy, s'installe en une commande et surpasse souvent Pandas sur les requêtes complexes.


## Plan

1. Installation et import
2. Base en mémoire vs persistée
3. Requêter des données Python
4. Requêter directement un fichier
5. SELECT, WHERE, ORDER BY, LIMIT
6. Agrégations et GROUP BY
7. Jointures
8. Window functions
9. CTE (WITH)
10. Intégration Pandas
11. Intégration Polars
12. Extensions
13. Performance sur Parquet
14. Synthèse
15. Exercices


---


## 1. Installation et import

`pip install duckdb`


In [ ]:
import duckdb
duckdb.__version__

---


## 2. Base en mémoire vs persistée


In [ ]:
con = duckdb.connect()  # base en mémoire

In [ ]:
# con = duckdb.connect('ma_base.duckdb')  # fichier sur disque

Comme SQLite, tout tient dans un fichier unique. On peut basculer en mémoire pour l'analyse ad hoc, ou sur disque pour persister.


---


## 3. Requêter des données Python

DuckDB peut requêter directement des `DataFrame` Pandas / Polars et des listes Python.


In [ ]:
import pandas as pd
df = pd.DataFrame({
    'nom': ['Alice', 'Bob', 'Chloé'],
    'age': [30, 25, 35],
})
duckdb.sql('SELECT * FROM df WHERE age > 28').df()

---


## 4. Requêter directement un fichier

L'innovation majeure de DuckDB : un chemin de fichier est un **nom de table** valide. Pas besoin d'importer.


In [ ]:
# duckdb.sql("SELECT * FROM 'data.csv'").df()

In [ ]:
# duckdb.sql("SELECT * FROM 'ventes.parquet' WHERE annee = 2024").df()

In [ ]:
# duckdb.sql("SELECT COUNT(*) FROM 'data/*.parquet'").df()  # glob pattern

---


## 5. SELECT, WHERE, ORDER BY, LIMIT


In [ ]:
con.execute('CREATE TABLE personnes (nom VARCHAR, age INTEGER, ville VARCHAR, salaire DOUBLE)')
con.execute("INSERT INTO personnes VALUES"
            " ('Alice', 30, 'Paris', 45000),"
            " ('Bob', 25, 'Lyon', 38000),"
            " ('Chloé', 35, 'Paris', 52000),"
            " ('Dimitri', 28, 'Nice', 41000)")
con.sql('SELECT * FROM personnes ORDER BY salaire DESC LIMIT 2').df()

---


## 6. Agrégations et GROUP BY


In [ ]:
con.sql('SELECT ville, AVG(salaire) AS moyen, COUNT(*) AS n FROM personnes GROUP BY ville').df()

---


## 7. Jointures


In [ ]:
con.execute('CREATE TABLE commandes (id INT, personne VARCHAR, montant DOUBLE)')
con.execute("INSERT INTO commandes VALUES (1, 'Alice', 200), (2, 'Bob', 150), (3, 'Alice', 300)")
con.sql('''
    SELECT p.nom, SUM(c.montant) AS ca
    FROM personnes p
    LEFT JOIN commandes c ON p.nom = c.personne
    GROUP BY p.nom
''').df()

---


## 8. Window functions


In [ ]:
con.sql('''
    SELECT nom, ville, salaire,
           RANK() OVER (PARTITION BY ville ORDER BY salaire DESC) AS rang_ville,
           AVG(salaire) OVER (PARTITION BY ville) AS salaire_moyen_ville
    FROM personnes
''').df()

---


## 9. CTE (`WITH`)


In [ ]:
con.sql('''
    WITH cte AS (
        SELECT ville, AVG(salaire) AS moyen
        FROM personnes
        GROUP BY ville
    )
    SELECT * FROM cte WHERE moyen > 40000
''').df()

---


## 10. Intégration Pandas

Le résultat d'une requête se convertit en DataFrame avec `.df()`, et on peut passer un DataFrame comme table.


In [ ]:
import pandas as pd
clients = pd.DataFrame({'id': [1, 2, 3], 'nom': ['A', 'B', 'C']})
duckdb.sql('SELECT nom FROM clients WHERE id > 1').df()

---


## 11. Intégration Polars


In [ ]:
# import polars as pl
# lf = pl.LazyFrame({'x': [1, 2, 3]})
# duckdb.sql('SELECT * FROM lf WHERE x > 1').pl()  # retourne un DataFrame Polars

DuckDB et Polars partagent le même format mémoire Arrow, donc les conversions sont **zero-copy**. Vous pouvez panacher librement.


---


## 12. Extensions


In [ ]:
# INSTALL httpfs; LOAD httpfs;  -- lire depuis S3/HTTP
# INSTALL spatial; LOAD spatial; -- GIS
# INSTALL icu;     LOAD icu;     -- collations Unicode

Avec `httpfs`, vous pouvez requêter directement un Parquet sur HTTP ou S3, sans téléchargement complet :
```sql
SELECT COUNT(*) FROM 'https://.../data.parquet';
```


---


## 13. Performance sur Parquet


DuckDB lit Parquet en **colonne** et exploite les **statistiques de min/max** par row group. Un `WHERE date > ...` peut sauter la majeure partie du fichier. C'est pour cela que la plupart des benchmarks donnent DuckDB en tête face à Pandas sur Parquet.


---


## Synthèse

| Notion | À retenir |
|---|---|
| DuckDB | base analytique embarquée, SQL sur fichiers |
| `duckdb.sql(...)` | exécute une requête, retourne un résultat |
| .df() / .pl() | convertit en DataFrame Pandas/Polars (zero-copy avec Polars) |
| Fichier = table | `SELECT * FROM 'file.parquet'` fonctionne directement |
| Window functions | `OVER (PARTITION BY ... ORDER BY ...)` |
| Extensions | `httpfs`, `spatial`, `icu`, `fts`... |


### Règles à retenir

- **Utiliser Parquet** pour les données persistées, pas CSV.
- **Requêter directement le fichier** quand il reste sur disque.
- **Laisser l'optimiseur pousser les prédicats** : filtrez au plus tôt en SQL.
- **`con.execute`** pour DDL/DML, **`con.sql`** pour obtenir un résultat.


---


## Exercices

Pour chaque exercice : écrivez une **fonction typée**, testez-la, puis comparez avec la correction. N'ouvrez pas la correction avant d'avoir vraiment cherché.


### Exercice 1 — Lire un CSV et compter

Écrivez `compter_lignes(chemin)` qui renvoie le nombre de lignes d'un CSV via DuckDB.


In [ ]:
import duckdb

def compter_lignes(chemin: str) -> int:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
import duckdb

def compter_lignes(chemin: str) -> int:
    res = duckdb.sql(f"SELECT COUNT(*) FROM '{chemin}'").fetchone()
    return int(res[0])
```

</details>


### Exercice 2 — Agrégation SQL sur un DataFrame

Étant donné un DataFrame Pandas `df` avec colonnes `categorie` et `prix`, renvoyez le prix moyen par catégorie en une requête SQL.


In [ ]:
import duckdb
import pandas as pd

def prix_moyen(df: pd.DataFrame) -> pd.DataFrame:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
import duckdb
import pandas as pd

def prix_moyen(df: pd.DataFrame) -> pd.DataFrame:
    return duckdb.sql('SELECT categorie, AVG(prix) AS moyen FROM df GROUP BY categorie').df()
```

</details>


### Exercice 3 — Top 3 par groupe

Écrivez une requête qui renvoie les 3 meilleures ventes **par catégorie** depuis une table `ventes(categorie, produit, ca)`.


In [ ]:
import duckdb

def top3(con: duckdb.DuckDBPyConnection) -> list:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
import duckdb

def top3(con: duckdb.DuckDBPyConnection) -> list:
    return con.sql('''
        SELECT categorie, produit, ca FROM (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY categorie ORDER BY ca DESC) AS r
            FROM ventes
        ) WHERE r <= 3
    ''').fetchall()
```

</details>


---


## Ressources externes

- [Documentation DuckDB](https://duckdb.org/docs/)
- [DuckDB Python API](https://duckdb.org/docs/api/python/overview)
- [DuckDB vs Pandas benchmark](https://duckdb.org/2021/05/14/sql-on-pandas.html)
